In [5]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [6]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline-2\\weights\\best.pt")

c:\Users\klanz\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn_augument.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_5356\1784273219.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\M

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [8]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [9]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [10]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [11]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [12]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [13]:
def run_yolo(image_path, conf=0.5):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [14]:
def run_pipeline(image_path, conf=0.5):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [15]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [16]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [17]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [18]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [19]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,420.2719,46.7440,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,17.8712,16.4539,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,15.2689,16.5190,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,CLOSE,14.8360,29.8110,"[0, 0, 0, 0, 0]","[0, 0, 0]","[0, 0, 1]"
4,1054.jpeg,OPEN,OPEN,OPEN,13.7982,26.1961,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,14.8920,17.7826,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,12.1710,18.1828,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,11.9250,17.0743,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,11.8494,15.3628,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,13.5813,25.0721,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [20]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [21]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [22]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [23]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9886
Precision: 0.9918
Recall   : 0.9877
F1-score : 0.9897

TP : 481
FP : 4
TN : 387
FN : 6

Średni czas: 14.35 ms


In [24]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9829
Precision: 0.9856
Recall   : 0.9836
F1-score : 0.9846

TP : 479
FP : 7
TN : 384
FN : 8

Średni czas: 21.69 ms


In [25]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.991753,0.987680,0.989712,481,4,387,6,14.346478
YOLO + CNN,0.982916,0.985597,0.983573,0.984584,479,7,384,8,21.694255


In [26]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [27]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [28]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 4
YOLO+CNN: 7


In [29]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,14.2545,17.4969,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,11.2336,38.5993,[1],[0],[1]


In [30]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [31]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 7


['coyote__lila_WSU_Lynx_IMG_0965_jpg.rf.vqQg0CptK1hvCGBkRVla.jpg',
 'Image-107-dbb8ca.jpg',
 'Image-23-0a5765.jpg',
 'Image-52-fd6d74.jpg',
 'Image-88-8f30f6.jpg',
 'neg_people__people_004_jpg.rf.zluE5eiBeb9pLTDNEBee.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [32]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 8


['1053.jpeg',
 '1085.jpeg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-eTv1aWFSXioPZgCY1NWdFgHaJ4.jpeg',
 'OIP-NEf6Uuinf5Z2jTEDnPLUHwHaFj.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg']

In [33]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-82-7d6856.jpg',
 'Image-84-bd2f1b.jpg',
 'Image-88-8f30f6.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [34]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-NEf6Uuinf5Z2jTEDnPLUHwHaFj.jpeg',
 'OIP-OLDB-DMfG6VSQCZwAyo_rAHaFj.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg']

In [35]:

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")



In [36]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [37]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [38]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [39]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [40]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [41]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,13.2065,11.9376,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,11.6820,17.3631,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,12.3945,16.7953,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,OPEN,11.9830,17.4761,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,OPEN,12.2615,23.4724,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,11.9957,16.3482,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,12.4823,20.4976,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,12.8688,15.7818,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,10.2765,15.5975,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,10.9542,22.3283,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [42]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,18.0187,12.3123,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,11.2964,17.6879,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,15.9504,17.8585,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,OPEN,15.2758,18.3618,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,OPEN,12.9255,23.4345,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,12.3117,15.6149,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,11.1051,17.2932,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,11.1480,16.0518,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,12.8312,16.7918,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,13.6903,19.6934,"[0, 0, 0]","[0, 0]","[0, 0]"


In [43]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,13.2501,11.0840,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,10.5244,16.1454,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,11.5328,16.2976,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,CLOSE,11.2822,23.9757,"[0, 0, 0, 0, 0]","[0, 0, 0]","[0, 1, 0]"
4,1054.jpeg,OPEN,OPEN,OPEN,11.5709,19.4796,"[0, 0, 0]","[0, 0]","[0, 0]"
5,1085.jpeg,OPEN,CLOSE,CLOSE,11.5159,10.7296,[0],[],[]
6,109.jpeg,OPEN,OPEN,OPEN,10.4940,14.7555,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,10.8117,16.0028,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,10.8454,14.9992,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,12.3450,23.8369,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [44]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,20.5349,28.6703,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,14.4880,19.0006,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,CLOSE,14.0953,20.2788,[0],[0],[1]
3,1053.jpeg,OPEN,OPEN,OPEN,13.9227,17.2619,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,12.7097,18.8445,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,13.0633,31.7490,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,23.1994,35.1860,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,25.2966,28.3494,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,17.8016,31.7521,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,21.9244,21.4748,"[0, 0, 0]",[0],[1]


In [45]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9795
Precision: 0.9896
Recall   : 0.9733
F1-score : 0.9814

TP : 474
FP : 5
TN : 386
FN : 13

Średni czas: 12.66 ms
pipeline
Accuracy : 0.9715
Precision: 0.9812
Recall   : 0.9671
F1-score : 0.9741

TP : 471
FP : 9
TN : 382
FN : 16

Średni czas: 20.53 ms


In [46]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9658
Precision: 0.9872
Recall   : 0.9507
F1-score : 0.9686

TP : 463
FP : 6
TN : 385
FN : 24

Średni czas: 13.86 ms
pipeline
Accuracy : 0.9556
Precision: 0.9766
Recall   : 0.9425
F1-score : 0.9592

TP : 459
FP : 11
TN : 380
FN : 28

Średni czas: 21.73 ms


In [47]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9487
Precision: 0.9911
Recall   : 0.9158
F1-score : 0.9520

TP : 446
FP : 4
TN : 387
FN : 41

Średni czas: 14.43 ms
pipeline
Accuracy : 0.9396
Precision: 0.9780
Recall   : 0.9117
F1-score : 0.9437

TP : 444
FP : 10
TN : 381
FN : 43

Średni czas: 23.19 ms


In [48]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9180
Precision: 0.9882
Recall   : 0.8624
F1-score : 0.9211

TP : 420
FP : 5
TN : 386
FN : 67

Średni czas: 13.50 ms
pipeline
Accuracy : 0.8371
Precision: 0.9649
Recall   : 0.7331
F1-score : 0.8331

TP : 357
FP : 13
TN : 378
FN : 130

Średni czas: 20.24 ms


In [49]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.979499,0.989562,0.973306,0.981366,474,5,386,13,12.661480
YOLO + CNN,0.971526,0.981250,0.967146,0.974147,471,9,382,16,20.530904


In [50]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.965831,0.987207,0.950719,0.968619,463,6,385,24,13.857465
YOLO + CNN,0.955581,0.976596,0.942505,0.959248,459,11,380,28,21.733116


In [51]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.948747,0.991111,0.915811,0.951974,446,4,387,41,14.432544
YOLO + CNN,0.939636,0.977974,0.911704,0.943677,444,10,381,43,23.193345


In [52]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.917995,0.988235,0.862423,0.921053,420,5,386,67,13.499407
YOLO + CNN,0.837130,0.964865,0.733060,0.833139,357,13,378,130,20.236858


In [53]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [54]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [55]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [56]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [57]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [58]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [59]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [60]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [61]:
dangerous_both1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") & (df_dark["yolo"]=="OPEN") & (df_dark["pipeline"]=="OPEN")
]
dangerous_both2 = df_night[
    (df_night["ground_truth"]=="CLOSE") & (df_night["yolo"]=="OPEN") & (df_night["pipeline"]=="OPEN")
]
dangerous_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") & (df_occlusion["yolo"]=="OPEN") & (df_occlusion["pipeline"]=="OPEN")
]
dangerous_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") & (df_motion_blur["yolo"]=="OPEN") & (df_motion_blur["pipeline"]=="OPEN")
]


In [62]:
comparison = pd.DataFrame(

    [   

        yolo_results,

        pipeline_results,

        yolo_results1,

        pipeline_results1,

        yolo_results2,

        pipeline_results2,

        yolo_results3,

        pipeline_results3,

        yolo_results4,

        pipeline_results4

    ],

    index=[

        "YOLO normal",

        "YOLO + CNN normal",

        "YOLO dark",

        "YOLO + CNN dark",

        "YOLO night",

        "YOLO + CNN night",

        "YOLO occlusion",

        "YOLO + CNN occlusion",

        "YOLO motion",

        "YOLO + CNN motion"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO normal,0.988610,0.991753,0.987680,0.989712,481,4,387,6,14.346478
YOLO + CNN normal,0.982916,0.985597,0.983573,0.984584,479,7,384,8,21.694255
YOLO dark,0.979499,0.989562,0.973306,0.981366,474,5,386,13,12.661480
YOLO + CNN dark,0.971526,0.981250,0.967146,0.974147,471,9,382,16,20.530904
YOLO night,0.965831,0.987207,0.950719,0.968619,463,6,385,24,13.857465
YOLO + CNN night,0.955581,0.976596,0.942505,0.959248,459,11,380,28,21.733116
YOLO occlusion,0.948747,0.991111,0.915811,0.951974,446,4,387,41,14.432544
YOLO + CNN occlusion,0.939636,0.977974,0.911704,0.943677,444,10,381,43,23.193345
YOLO motion,0.917995,0.988235,0.862423,0.921053,420,5,386,67,13.499407
YOLO + CNN motion,0.837130,0.964865,0.733060,0.833139,357,13,378,130,20.236858


In [63]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"     YOLO night:",len(dangerous_yolo2),"     YOLO occlusion:",len(dangerous_yolo3),"    YOLO motion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN motion:",len(dangerous_pipeline4))
print("BOTH dark:",len(dangerous_both1),"     BOTH night:",len(dangerous_both2),"     BOTH occlusion:",len(dangerous_both3),"    BOTH motion:",len(dangerous_both4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 5      YOLO night: 6      YOLO occlusion: 4     YOLO motion: 5
YOLO+CNN dark: 9 YOLO+CNN night: 11 YOLO+CNN occlusion: 10 YOLO+CNN motion: 13
BOTH dark: 2      BOTH night: 2      BOTH occlusion: 2     BOTH motion: 1


In [74]:
locking_chicken1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE") | (df_dark["pipeline"]=="CLOSE"))
]
locking_chicken2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE") | (df_night["pipeline"]=="CLOSE"))
]
locking_chicken3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE") | (df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE") | (df_motion_blur["pipeline"]=="CLOSE"))
]

In [75]:
print()
print("Niekrytyczne błędy, niewpuszczenie kur")
print("----------------")
print("YOLO dark:",len(locking_chicken1),"     YOLO night:",len(locking_chicken2),"     YOLO occlusion:",len(locking_chicken3),"    YOLO motion:",len(locking_chicken4))



Niekrytyczne błędy, niewpuszczenie kur
----------------
YOLO dark: 18      YOLO night: 29      YOLO occlusion: 43     YOLO motion: 131


In [64]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,12.8767,17.5083,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,9.5207,16.1974,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,12.1371,18.3482,[1],[0],[1]


In [65]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,13.6309,18.6985,[1],[0],[1]
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,13.9153,18.1791,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,10.7151,16.6281,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,13.1432,16.5891,[1],[0],[1]


In [66]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,13.1565,18.8411,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,11.6606,18.1305,[1],[0],[1]


In [67]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,13.0999,17.7406,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,12.3362,16.1664,[1],[0],[1]
872,raptor__raptor_012_jpg.rf.yRUSw4eHaOfcUqIhXQVM...,CLOSE,OPEN,CLOSE,12.1517,18.1541,"[1, 1]",[0],[1]
873,raptor__raptor_014_jpg.rf.WcLMSucKo0EGCfOoYu3y...,CLOSE,OPEN,CLOSE,11.9638,18.2797,"[1, 1]",[0],[1]


In [68]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
89,coyote__lila_AMMonitor_Camera_Traps_MMP-Suc2_0...,CLOSE,CLOSE,OPEN,16.1270,23.8451,[1],[1],[0]
116,coyote__lila_Felidae_Conservation_Fund_2020-20...,CLOSE,CLOSE,OPEN,21.9804,26.2747,[1],[1],[0]
142,coyote__lila_WSU_Lynx_IMG_0965_jpg.rf.vqQg0Cpt...,CLOSE,CLOSE,OPEN,12.0896,20.2417,[1],[1],[0]
208,fox__lila_Snapshot_Serengeti_S2_I13_R1_PICT100...,CLOSE,CLOSE,OPEN,12.0310,15.7047,"[1, 1]",[1],[0]
220,Image-107-dbb8ca.jpg,CLOSE,CLOSE,OPEN,14.4269,25.3114,[1],[1],[0]
240,Image-23-0a5765.jpg,CLOSE,CLOSE,OPEN,14.2350,17.8537,[1],[1],[0]
264,Image-42-dd926b.jpg,CLOSE,CLOSE,OPEN,26.3667,44.6240,[1],[1],[0]
271,Image-46-c30e49.jpg,CLOSE,CLOSE,OPEN,13.9731,18.1960,[1],[1],[0]
332,Image-84-a5ad3f.jpg,CLOSE,CLOSE,OPEN,16.2951,19.4173,[1],[1],[0]
339,Image-88-8f30f6.jpg,CLOSE,CLOSE,OPEN,10.9515,14.1439,[1],[1],[0]


In [69]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [70]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [71]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [72]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [73]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!